# 01.1 Rules or Learning? The Decision That Precedes the Model

> **Prerequisites:** none — this is the track's first notebook. Assumed: working Python and
> backend experience (see `CLAUDE.md` learner profile).
> **What you'll learn:**
> - Measure a hand-written business rule as a model, so you have a floor before you train anything
> - Compare a learned model against that floor at a matched operating point instead of at a flattering one
> - Tell a metric win apart from a business win, and both apart from run-to-run noise
> - Diagnose a model whose monitored metric improves while the outcome it serves degrades
> - Decide, with numbers, whether an ML system is worth its operating cost at all
> **Level:** Beginner · **Series:** 01 The ML Landscape & Project Lifecycle

> ⚡ **Tuesday 2025-07-15, 09:40** — six weeks after PayFlow's payment gateway migration, the
> collections dashboard is green: the late-payment model's precision has climbed from 0.472 to
> 0.570. Meanwhile overdue balance is climbing and the CFO wants to know why nobody is chasing
> the invoices that went bad. The cause: the model's score distribution never moved, so the
> queue never grew — while the world it was trained on quietly changed underneath it.

## Concept
### Plain-English Explanation

PayFlow's collections team has a finite amount of attention. Roughly twenty-six thousand
invoices go out in a five-month window and an analyst can chase only a fraction of them, so
somebody has to decide which ones get chased. For years that decision was three clauses
written on a whiteboard: chase the big invoices, chase the enterprise accounts, chase the Gulf
accounts. The obvious modern instinct is to replace that whiteboard with a trained model.

The question this notebook answers is not *how* to train that model. It is whether you should,
and how you would know. Those three whiteboard clauses are already a model: they take features
about an invoice, apply parameters a human chose, and emit a decision. They cost nothing to
serve, never drift, and can be explained to an auditor in one sentence. A learned model has to
beat that — not in the abstract, and not on a metric chosen after the fact, but on the number
the business actually cares about, by a margin bigger than the noise in your own experiment.

### Technical Explanation

The task is **binary classification** framed as a **ranking under a capacity constraint**. For
each invoice we want the probability it will be paid more than seven days after its due date,
and we can act on only the top *k*, where *k* is what the team can work. That capacity framing
matters more than it first appears: it means accuracy is meaningless here (predicting "never
late" is correct on the 1 − 0.297 of invoices that are not late, and produces an empty queue),
and it means any two policies must be compared at the *same k*, or you are comparing appetite
rather than skill.

Three quantities carry the comparison. **Precision@k** is the share of the chased invoices that
really were late — what fraction of analyst effort was well spent. **Recall@k** is the share of
all late invoices the queue caught. And **value captured** is the money that policy recovers,
modelled as the working-capital cost avoided by being paid sooner: for each late invoice caught,
40% of its lateness is avoided, priced at a 12% annual cost of capital. That third quantity is
the objective; the first two are proxies for it, and a large part of this notebook is about the
gap between a proxy and an objective.

The **base rate** — the share of resolved invoices that are late — is 0.297 overall, and it is
the number every claim gets measured against. A policy that flags a random 26.4% of invoices
achieves a precision equal to the base rate, and that random policy is the true zero point.
When someone reports a model at 0.446 precision, the honest question is not whether 0.446 is
good but what the rule scored and what random scored on the same rows.

⭐ **CRITICAL CONCEPT** — the features must be knowable at the moment the decision is made.
PayFlow's invoice export carries `reminder_count`, which is a superb predictor of lateness and
is written *after* the payment resolves; the customer table carries
`total_lifetime_value_usd`, computed over all time including the future. Both are recorded in
`_data/SPEC.md` as leakage traps (M10) and both are excluded here. A model trained with them
looks brilliant offline and collapses in production, because at scoring time those columns hold
values the training data never contained.

### Mental Model

A business rule is a model with hand-set parameters, zero training cost, and zero drift. Machine
learning buys you a better decision boundary in exchange for a permanent operating obligation —
retraining, monitoring, on-call, an incident budget. Ask what the boundary is worth in the
business's own units before you sign up for the obligation.

## How It Works

An invoice becomes a row the moment it is issued, and everything the decision can use must be
knowable at that instant. That constraint defines the feature set and it is also where most
first ML projects quietly fail.

```text
                       PayFlow invoice at issue time
                                   |
        +--------------------------+---------------------------+
        | knowable now: segment, country, amount_usd, terms,    |
        |               seats, tenure_days, plan, industry      |
        | NOT knowable: gateway, fx at payment, reminder_count  |  <- SPEC M10 leakage
        +--------------------------+---------------------------+
                                   |
              +--------------------+---------------------+
              v                                          v
   RULE: three hand-set thresholds          MODEL: fitted coefficients over
   amount_usd>1500 OR Enterprise OR AE      the same features, amount-blind
              |                                          |
              +--------------------+---------------------+
                                   v
                    queue of k invoices per period  (capacity, not choice)
                                   v
        value = SUM over caught late invoices of
                0.40 * days_late * amount_usd * (0.12/365)
```

Read the bottom two boxes together, because their mismatch is the whole notebook. The queue is
selected by *probability of lateness*. The value is earned in *dollars of lateness*. A late
$90 invoice and a late $9,000 invoice are worth the same to a classifier trained on a 0/1
label, and are worth a hundred times different amounts to PayFlow. Nothing in the standard
training objective knows that, and nothing in a precision dashboard will tell you.

The rule, written by someone who had chased invoices for a living, encodes the money directly:
its first clause is a threshold on amount. It was never trying to estimate a probability. It
was trying to protect revenue, which is a different objective and — as the numbers below show —
a better-aligned one.

The drift mechanism matters too. On 2025-07-01 PayFlow migrated non-Indian customers from
Stripe to Adyen, and settlement for those customers slowed by roughly four days (`_data/SPEC.md`
rule M9). This is **label shift**: the relationship between an invoice's features and its
lateness probability changed, while the features themselves look completely normal. No null
rate moves, no schema check fires, no input distribution shifts. A model trained before the
migration keeps emitting pre-migration probabilities against a post-migration world.

## Hands-On Build
### Stage A — from scratch

Before any model, the floor. The lab module owns the messy-CSV plumbing — duplicate invoice
rows, comma-formatted amounts, two incompatible timestamp formats, mixed currencies — because
cleaning is series 09's subject, not this notebook's. What it returns is the modelling table
and an audit of what it had to survive.

In [1]:
# The committed lab script owns data construction; 01.1 consumes it. Every number below is
# reproduced by: .venv\Scripts\python "01-ml-landscape-and-lifecycle/_lab/lab_01.1_rules_vs_learning.py"
import importlib.util
import sys
from pathlib import Path

import numpy as np
import pandas as pd

LAB = Path.cwd() / "_lab" / "lab_01.1_rules_vs_learning.py"
spec = importlib.util.spec_from_file_location("lab_01_1", LAB)
lab = importlib.util.module_from_spec(spec)
sys.modules["lab_01_1"] = lab
spec.loader.exec_module(lab)

df, audit = lab.build_dataset()      # PayFlow invoices x payments x customers
w = lab.windows(df)                  # temporal split: train / test_stable / test_drift

for key, value in audit.items():
    print(f"{key:<34} {value:>9,}")
print(f"{'late rate (all resolved)':<34} {df['late'].mean():>9.3f}")
for name, part in w.items():
    print(f"{name:<12} n={len(part):>7,}  late rate={part['late'].mean():.3f}  "
          f"median days_late={part['days_late'].median():.0f}")

invoices_raw                         288,936
payments_raw                         286,432
exact_duplicate_rows_dropped             896
comma_formatted_amounts                3,789
glitch_window_rows_excluded              676
payments_lost_to_naive_iso_parse      43,165
resolved_invoices                    274,849
late rate (all resolved)               0.297
train        n=159,821  late rate=0.262  median days_late=3
test_stable  n= 27,290  late rate=0.263  median days_late=3
test_drift   n= 68,169  late rate=0.389  median days_late=5


Two lines in that audit deserve more attention than the model will get. `payments_lost_to_naive_iso_parse`
is 43,165 — the legacy gateway writes `DD/MM/YYYY HH:MM` while everything else writes ISO-8601,
so parsing with a single format silently discards fifteen percent of all payments. That is not
a rounding error, it is a biased sample: the discarded rows are exactly one gateway's customers.
And `exact_duplicate_rows_dropped` is 896, from an invoicing repost bug. Neither of these would
raise an exception. Both would change every number that follows.

Note also that the training window and the stable test window have almost identical late rates,
0.262 and 0.263, while the drift window sits at 0.389. Hold that third number; it is the
incident.

Now the rule. Three clauses, no fitting, and the honest comparison partner: a random policy that
flags the same number of invoices.

In [2]:
def dunning_rule(d: pd.DataFrame) -> np.ndarray:
    """Collections' whiteboard policy, unchanged since 2024.

    "Chase the big ones, the enterprises, and the Gulf accounts." Three hand-set thresholds.
    No training data, no fitting, no drift, and one sentence of explanation for an auditor.
    """
    return ((d["amount_usd"] > 1500)
            | (d["segment"] == "Enterprise")
            | (d["country"] == "AE")).to_numpy()


stable = w["test_stable"]                 # 2025-01..05, before the gateway migration
y = stable["late"].to_numpy()
rule_flag = dunning_rule(stable)
k = int(rule_flag.sum())                  # the queue size the team can actually work
rng = np.random.default_rng(42)

print(f"base rate = {y.mean():.3f}   queue k = {k:,} ({k / len(stable):.1%} of invoices)\n")
lab.report("random at same k", stable, lab.topk_flag(rng.random(len(stable)), k))
lab.report("Stage A: dunning rule", stable, rule_flag)
lab.report("work everything", stable, np.ones(len(stable), dtype=bool))

base rate = 0.263   queue k = 7,195 (26.4% of invoices)

  random at same k           k= 7,195 (26.4%)  prec=0.265  rec=0.265  value=$  14,384  net=$   -3,604
  Stage A: dunning rule      k= 7,195 (26.4%)  prec=0.425  rec=0.426  value=$  51,362  net=$   33,374
  work everything            k=27,290 (100.0%)  prec=0.263  rec=1.000  value=$  54,589  net=$  -13,636


The random policy scores 0.265 precision, which is the base rate, as it must be — that is the
definition of learning nothing. The rule scores 0.425, sixty percent better than random, and
captures $51,362 of value against random's $14,384. A whiteboard beat chance by a wide margin,
which is unsurprising once you notice that a person who chases invoices for a living has been
running an unlicensed feature-selection process for years.

The `work everything` row is the other bound and it is the one people forget. Chasing all 27,290
invoices captures the most value in absolute terms, $54,589, and loses $13,636 net, because the
cost of the queue exceeds what the marginal invoices return. Capacity is not an annoyance to be
engineered away; it is what makes the ranking problem a problem.

### Stage B — idiomatic

Same rows, same operating point, now with scikit-learn. The pipeline is the unit of work —
imputation, scaling and encoding live *inside* it so that nothing fitted on training rows can
touch the evaluation rows. The feature lists come from the lab and deliberately exclude the two
leakage columns.

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUM, CAT = lab.NUM, lab.CAT     # reminder_count and lifetime value are NOT here (SPEC M10)

model = Pipeline([
    ("prep", ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("sc", StandardScaler())]), NUM),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore"))]), CAT)])),
    ("clf", LogisticRegression(max_iter=1000, random_state=42)),
])
model.fit(w["train"][NUM + CAT], w["train"]["late"])
scores = model.predict_proba(stable[NUM + CAT])[:, 1]
model_flag = lab.topk_flag(scores, k)          # top-k, so the queue size matches the rule's

lab.report("Stage A: dunning rule", stable, rule_flag)
lab.report("Stage B: logistic reg", stable, model_flag)
lab.report("Stage B': grad boosting", stable,
           lab.topk_flag(lab.fit_score(w["train"], stable, factory=lab.make_boosted), k))
print(f"\nmean amount of the late invoices each policy catches:  "
      f"rule ${stable.loc[rule_flag & (stable['late'] == 1), 'amount_usd'].mean():,.0f}   "
      f"model ${stable.loc[model_flag & (stable['late'] == 1), 'amount_usd'].mean():,.0f}")

  Stage A: dunning rule      k= 7,195 (26.4%)  prec=0.425  rec=0.426  value=$  51,362  net=$   33,374
  Stage B: logistic reg      k= 7,195 (26.4%)  prec=0.446  rec=0.446  value=$  49,665  net=$   31,677


  Stage B': grad boosting    k= 7,195 (26.4%)  prec=0.453  rec=0.454  value=$  50,549  net=$   32,561

mean amount of the late invoices each policy catches:  rule $6,178   model $5,631


⚠️ Read those three rows twice, because they disagree with each other. The logistic regression
wins on precision, 0.446 against the rule's 0.425. Gradient boosting wins by more, 0.453 — so
the gap is not an artifact of picking a weak model class. And both of them **lose on money**:
$49,665 and $50,549 against the rule's $51,362.

The last line explains the reversal. The late invoices the rule catches average $6,178; the ones
the model catches average $5,631. The rule's first clause is a threshold on amount, so it is
hunting money by construction. The model was trained on a 0/1 label in which a late $90 invoice
and a late $9,000 invoice are the same event, so it is hunting probability and is blind to size.
Precision counts invoices. The business counts dollars. Optimizing the first can move you away
from the second, and no amount of model capacity fixes a misspecified objective — that is what
the gradient-boosting row proves.

### Stage C — production

If this rule is going to keep beating the model on the objective, then the production design is
not "deploy the model." It is: the rule is the floor, the model must earn its slot on every run,
and the system falls back automatically when the model stops looking healthy. The guard below
compares what the model predicts against the recently observed rate — the exact check that would
have caught the incident this notebook opens with.

In [4]:
from dataclasses import dataclass


@dataclass(frozen=True)
class QueueDecision:
    invoice_ids: list[str]
    policy: str
    reason: str


def build_dunning_queue(day: pd.DataFrame, scores: np.ndarray | None, capacity: int,
                        *, recent_actual_rate: float | None = None,
                        max_calibration_gap: float = 0.05) -> QueueDecision:
    """Select today's dunning queue. The rule is the floor AND the fallback.

    The model is used only while it stays calibrated against recently observed lateness;
    otherwise we serve the rule, which cannot drift because it never learned anything.
    """
    rule = dunning_rule(day)
    if scores is None:                                     # no artifact / failed load
        return QueueDecision(day.loc[rule, "invoice_id"].tolist()[:capacity], "rule",
                             "no model artifact available")
    if recent_actual_rate is not None:
        gap = abs(recent_actual_rate - float(scores.mean()))
        if gap > max_calibration_gap:                      # the incident's tripwire
            return QueueDecision(day.loc[rule, "invoice_id"].tolist()[:capacity], "rule",
                                 f"calibration gap {gap:.3f} exceeds {max_calibration_gap}")
    picked = lab.topk_flag(scores, min(capacity, len(day)))
    return QueueDecision(day["invoice_id"].to_numpy()[picked].tolist(), "model", "healthy")


day = w["test_drift"].head(5000)
day_scores = lab.fit_score(w["train"], day)
healthy = build_dunning_queue(day, day_scores, 300, recent_actual_rate=0.27)
tripped = build_dunning_queue(day, day_scores, 300,
                              recent_actual_rate=float(day["late"].mean()))
print(f"healthy run -> {healthy.policy:<5} ({healthy.reason}), {len(healthy.invoice_ids)} queued")
print(f"drifted run -> {tripped.policy:<5} ({tripped.reason}), {len(tripped.invoice_ids)} queued")

healthy run -> model (healthy), 300 queued
drifted run -> rule  (calibration gap 0.132 exceeds 0.05), 300 queued


The same scores produce two different policies depending on one piece of context the model
itself does not have: what actually happened recently. That is the shape of every honest ML
deployment — the model is a component inside a system that is allowed to distrust it, and the
thing it falls back to is the baseline you were told to build first.

## Evaluation

The metric is precision@k and the objective is value captured, both measured at the rule's own
queue size on the pre-migration window. The harness is eight bootstrap refits of the training
set, which gives a spread to compare deltas against; without that spread, "the model is better"
is an opinion. Logistic regression is deterministic given data, so the variance being measured
here is sampling variance in the training set — the honest question of whether this result would
survive a slightly different history.

In [5]:
precs, vals, weighted_vals = [], [], []
for s in range(8):                                    # bootstrap the TRAINING set, refit, rescore
    boot = w["train"].sample(frac=1.0, replace=True, random_state=s)
    plain = lab.topk_flag(lab.fit_score(boot, stable, seed=s), k)
    wtd = lab.topk_flag(lab.fit_score(boot, stable, seed=s, weight_by_value=True), k)
    precs.append(lab.precision_recall_at_k(y, plain)["precision"])
    vals.append(lab.value_captured(stable, plain))
    weighted_vals.append(lab.value_captured(stable, wtd))

rule_p = lab.precision_recall_at_k(y, rule_flag)["precision"]
rule_v = lab.value_captured(stable, rule_flag)


def verdict(delta: float, band: float) -> str:
    return "SIGNAL" if abs(delta) > band else "NOISE"


print(f"precision   model {np.mean(precs):.4f} +/- {np.std(precs):.4f}   rule {rule_p:.4f}   "
      f"delta {np.mean(precs) - rule_p:+.4f}   noise band {2 * np.std(precs):.4f} -> "
      f"{verdict(np.mean(precs) - rule_p, 2 * np.std(precs))}")
print(f"value       model ${np.mean(vals):,.0f} +/- ${np.std(vals):,.0f}   rule ${rule_v:,.0f}   "
      f"delta ${np.mean(vals) - rule_v:+,.0f}   noise band ${2 * np.std(vals):,.0f} -> "
      f"{verdict(np.mean(vals) - rule_v, 2 * np.std(vals))}")
print(f"value(wtd)  model ${np.mean(weighted_vals):,.0f} +/- ${np.std(weighted_vals):,.0f}   "
      f"rule ${rule_v:,.0f}   delta ${np.mean(weighted_vals) - rule_v:+,.0f}   "
      f"noise band ${2 * np.std(weighted_vals):,.0f} -> "
      f"{verdict(np.mean(weighted_vals) - rule_v, 2 * np.std(weighted_vals))}")

precision   model 0.4448 +/- 0.0008   rule 0.4254   delta +0.0194   noise band 0.0015 -> SIGNAL
value       model $49,454 +/- $218   rule $51,362   delta $-1,908   noise band $437 -> SIGNAL
value(wtd)  model $51,134 +/- $63   rule $51,362   delta $-228   noise band $126 -> SIGNAL


The precision delta is +0.0194 against a noise band of 0.0015, so the model genuinely ranks
better than the rule; that is not a fluke of one training sample. The value delta is −$1,908
against a band of $437, so the model genuinely captures less money. Both statements are
supported at the same time, and the second one is the one that pays salaries.

The third row is the payoff. Re-fitting the *same* algorithm on the *same* features with
`sample_weight=amount_usd` — one argument, no new data, no new model class — moves value
captured from $49,454 to $51,134 and closes most of the gap to the rule's $51,362. The
remaining delta is −$228 against a band of $63. Formally that is still outside the noise band,
and it is worth being precise about what that means: the difference is statistically detectable
and practically nil, under half a percent of the objective. Statistical significance and
practical significance are different claims, and conflating them is how teams end up shipping
systems that win a t-test and lose money.

So the honest verdict for PayFlow today is that a well-specified model draws level with the
whiteboard and does not beat it, while adding a retraining schedule, a monitoring surface, and
the on-call burden documented below. That is a legitimate reason not to ship one — and it is
also the measurement that tells you exactly what would have to change for the answer to flip.

## Design Patterns / Tradeoffs

**Hand-written rule.** Parameters are set by a domain expert. Serving cost is a boolean
expression; there is no artifact to version, no training data to retain, no drift, and the
explanation to a regulator or a customer is one sentence. It encodes the objective directly
when the expert thinks in the objective's units, which is why this one hunts large amounts. Its
failure mode is that it cannot represent interactions or thresholds nobody thought of, it goes
stale silently when the business changes, and every edit is a negotiation with the person who
wrote it. Use it when the signal lives in a handful of features a domain expert already knows,
when explainability is a hard requirement, or when you have no labelled history yet. Do not use
it when the boundary is genuinely multivariate, or when the number of clauses has grown past
what one person can hold in their head — a forty-clause rule is an untested, unversioned model.

**Learned classifier.** Parameters are fitted, so it finds interactions nobody specified, adapts
by retraining, and produces a calibrated score you can threshold to trade precision against
recall. The costs are structural rather than incidental: it needs labelled history with the
outcome you actually care about, a feature pipeline that is identical in training and serving,
version control over artifacts, monitoring for drift, and somebody carrying a pager. It also
optimizes the objective you wrote down, not the one you meant. Use it when signal is diffuse
across many weak features, when the boundary is nonlinear, or when the operating point needs to
move continuously. Do not use it as the first move on a problem where three thresholds capture
most of the available signal — which is exactly the situation measured above.

**Recommendation for PayFlow:** keep the rule as the production policy and keep the
value-weighted model in shadow mode, scoring every invoice and logging its queue without acting
on it. Revisit when any one of three conditions holds: the shadow model's value captured exceeds
the rule's by more than the bootstrap band for four consecutive weeks; the feature set grows to
include something the rule cannot express, such as behavioural signals from support tickets or
payment history; or the queue capacity changes materially, since the whole comparison is
conditioned on k.

## Production Scenario
### Symptoms

**Tuesday 2025-07-15, 09:40.** The model has been live since March, selecting the daily dunning
queue by thresholding its predicted probability at 0.35 — a threshold chosen on 2024 validation
data and never revisited. The gateway migration completed on 2025-07-01.

What the on-call sees, in the order they see it:

- **09:40** — a finance alert, not an ML alert: `dso_days` (days sales outstanding) has risen
  for eleven consecutive days and crossed its quarterly threshold. Overdue balance is up.
- The collections model dashboard is **green**. Rolling precision has *improved*, from 0.472
  before the migration to 0.570 after it. Nobody paged the ML team, because by the ML team's own
  metric nothing is wrong.
- The daily queue size looks normal: 20.1% of invoices flagged, against 21.2% before. Analysts
  report they are getting through the queue comfortably — which, in hindsight, was the first
  real symptom.
- Ticket volume from the collections team mentions invoices "going bad without ever showing up
  on the list."

In [6]:
THRESHOLD = 0.35     # chosen on 2024 validation data, shipped in March, never revisited
drift = w["test_drift"]
scores_drift = lab.fit_score(w["train"], drift)

for label, part, sc in [("stable 2025-01..05", stable, scores),
                        ("drift  2025-07..2026-05", drift, scores_drift)]:
    yy = part["late"].to_numpy()
    m = lab.precision_recall_at_k(yy, sc >= THRESHOLD)
    print(f"{label:<24} predicted={sc.mean():.3f}  ACTUAL={yy.mean():.3f}  "
          f"queue={m['queue_share']:5.1%}  precision={m['precision']:.3f}  "
          f"recall={m['recall']:.3f}")

print(f"\ncalibration gap  {y.mean() - scores.mean():+.3f} -> "
      f"{drift['late'].mean() - scores_drift.mean():+.3f}")
# Split the label by the population that changed systems - the decisive diagnostic step.
for label, in_stable, in_drift in [
        ("IN     (razorpay, untouched)", stable["country"] == "IN", drift["country"] == "IN"),
        ("non-IN (migrated to adyen)", stable["country"] != "IN", drift["country"] != "IN")]:
    print(f"median days_late {label}: {stable.loc[in_stable, 'days_late'].median():.0f}"
          f" -> {drift.loc[in_drift, 'days_late'].median():.0f}")

stable 2025-01..05       predicted=0.268  ACTUAL=0.263  queue=21.2%  precision=0.472  recall=0.381
drift  2025-07..2026-05  predicted=0.264  ACTUAL=0.389  queue=20.1%  precision=0.570  recall=0.294

calibration gap  -0.004 -> +0.125
median days_late IN     (razorpay, untouched): 5 -> 5
median days_late non-IN (migrated to adyen): 2 -> 6


### Diagnosis

Walking the ML observability ladder, and naming which signal eliminated which hypothesis:

1. **Alert** — `dso_days` rising. Business symptom, no ML attribution yet. Candidate causes:
   more late invoices, a smaller queue, a worse queue, or a collections staffing change.
2. **Model-quality dashboard** — rolling precision 0.472 → 0.570, i.e. *better*. This is what
   made the incident last six weeks. It eliminates "the queue got worse at what it does" and
   should immediately raise suspicion of the dashboard itself, because precision rises
   mechanically when the positive class becomes more common.
3. **Prediction logs** — the score distribution is unmoved: mean predicted lateness 0.268 before,
   0.264 after. Meanwhile the observed rate is 0.263 before and 0.389 after. The calibration gap
   goes from −0.004 to +0.125. This is the smoking gun: the model is describing a world that no
   longer exists. It also explains the queue that never grew, since a fixed threshold on an
   unmoved distribution flags a near-constant share, 21.2% → 20.1%, while the number of genuinely
   late invoices rose by roughly half. Recall falls from 0.381 to 0.294.
4. **Input-data checks** — schema unchanged, null rates unchanged, amount and tenure
   distributions unchanged. This eliminates the usual first suspect, a broken upstream feed, and
   is why the pipeline's own validation stayed quiet.
5. **Drift analysis** — split the label by country. Indian customers' median days-late is
   unchanged at 5; non-Indian customers move from 2 to 6. The shift is localized to exactly the
   population that changed payment gateway. This is label shift with stationary features:
   nothing about the inputs moved, only the mapping from inputs to outcome.
6. **Version diff** — no model retrain, no code deploy, no feature change in the window. The
   only change in the blast radius is the 2025-07-01 gateway migration, a payments-team project
   that never appeared on the ML team's radar because it touched no ML system.

### Root Cause

The Stripe-to-Adyen migration lengthened settlement for non-Indian customers by about four days,
raising the true late rate from 0.263 to 0.389 while leaving every input feature statistically
unchanged. The model, trained entirely on pre-migration invoices, kept emitting pre-migration
probabilities, so a fixed decision threshold selected a queue of roughly constant size against a
population with far more late invoices in it — and precision, the monitored metric, *rose*
because the positive class had become denser.

### Fix

**Mitigation now (same day).** Flip the queue to the rule via the fallback in Stage C, and size
the queue by the recently observed late rate rather than by the model's predicted rate. The rule
does not need to be trusted or retrained, and on post-migration invoices it outscores the stale
model at a matched queue size — 0.547 against 0.521 in the cell below — so the mitigation is an
upgrade rather than a concession.

**Permanent fix.** Retrain on data that contains the new regime, and — because the earlier
evaluation showed a probability ranker is not aligned with the objective — retrain with
`sample_weight=amount_usd`. Then add the calibration guard from Stage C as a hard gate in the
scoring job, so a future regime change degrades to the rule automatically instead of degrading
silently. The cell below quantifies what the retrain does and does not buy.

In [7]:
eval_w = df.loc[(df["issue_date"] >= "2026-01-01") & (df["issue_date"] < "2026-05-31")]
post_migration = df.loc[(df["issue_date"] >= lab.MIGRATION) & (df["issue_date"] < "2026-01-01")]
kk = int(dunning_rule(eval_w).sum())

stale = lab.fit_score(w["train"], eval_w)                 # trained pre-2025-01
fresh = lab.fit_score(post_migration, eval_w)             # trained on the new regime only

print(f"eval window 2026-01..05  n={len(eval_w):,}   actual late rate={eval_w['late'].mean():.3f}")
print(f"stale model predicts {stale.mean():.3f}  -> gap {eval_w['late'].mean() - stale.mean():+.3f}")
print(f"refit model predicts {fresh.mean():.3f}  -> gap {eval_w['late'].mean() - fresh.mean():+.3f}\n")
lab.report("rule", eval_w, dunning_rule(eval_w))
lab.report("model (stale)", eval_w, lab.topk_flag(stale, kk))
lab.report("model (refit on regime)", eval_w, lab.topk_flag(fresh, kk))

eval window 2026-01..05  n=32,768   actual late rate=0.390
stale model predicts 0.266  -> gap +0.124
refit model predicts 0.372  -> gap +0.018

  rule                       k= 8,425 (25.7%)  prec=0.547  rec=0.361  value=$  71,628  net=$   50,565
  model (stale)              k= 8,425 (25.7%)  prec=0.521  rec=0.344  value=$  66,673  net=$   45,611
  model (refit on regime)    k= 8,425 (25.7%)  prec=0.557  rec=0.367  value=$  68,970  net=$   47,907


Retraining repairs calibration decisively — the gap collapses from +0.124 to +0.018, which
restores the queue-sizing behaviour that broke — and it lifts precision from 0.521 to 0.557,
now ahead of the rule's 0.547. On value captured the refit model reaches $68,970 against the
rule's $71,628, so it is still behind on the objective. Retraining fixed the level and part of
the order; it did not fix the specification. This is the distinction worth carrying: drift is
repaired by data, misalignment is repaired by rewriting the objective.

### Prevention

- **Calibration monitor**, not just a precision monitor: alert when the rolling gap between mean
  predicted rate and observed rate exceeds 0.05. This fires on the real symptom and would have
  paged on roughly 2025-07-14, six weeks earlier than the finance alert did.
- **The baseline runs forever.** Score the rule in shadow on every batch and alert when
  `model_value − rule_value` goes negative for four consecutive weeks. A baseline evaluated once,
  offline, before launch is a decoration; a baseline evaluated continuously is a canary.
- **Monitor recall against a delayed ground truth**, not only precision. Precision is
  computable from the queue alone and rises under label shift; recall requires knowing about the
  invoices you did not chase, which is precisely the blind spot this incident lived in.
- **Register infrastructure changes as ML events.** The gateway migration was a payments-team
  ticket with no ML system in its blast radius, and it invalidated a production model. Any change
  to a system that generates the label-forming process belongs on the model owner's calendar.

## Common Pitfalls

**Reporting a model score with no baseline.** "The churn model hits 0.446 precision" is not a
result; 0.446 against a random baseline of 0.265 and a rule at 0.425 is. The mechanism is that
model quality and problem difficulty are confounded in a single number, and only the comparison
separates them. Always publish the trio: random, incumbent, model.

**Comparing policies at different operating points.** A model evaluated on its most flattering
threshold against a rule evaluated at its natural flag rate is not a comparison. Fix k, or fix
the threshold, or plot the whole curve — but never compare one policy's best point against
another's arbitrary point.

⚠️ **Optimizing a proxy and reporting the proxy.** The clean trap of this notebook: precision
improved and money fell, and both facts were true simultaneously. Whenever the business value of
a correct prediction varies across rows — invoice amount, customer size, claim severity — an
unweighted classifier is optimizing something the business did not ask for.

**Treating accuracy as informative on imbalanced data.** With a base rate of 0.297, predicting
"never late" is correct on seven of every ten invoices and produces an empty queue — a number
that looks respectable in a status update and describes a system that does nothing. Series 15
makes this canonical; the habit starts here.

⚠️ **Monitoring only metrics that are computable from the model's own output.** Precision over
the chased queue needs no counterfactual and therefore drifts upward under label shift, which is
why the dashboard was green for six weeks. Any monitoring set consisting only of self-computable
metrics has this hole.

**Letting a leakage column in because it improves the score.** `reminder_count` would lift every
number above and is written after the outcome resolves. The tell is a feature that is
"too good"; the discipline is asking, for every column, what its value would be at the instant
the decision is made. Series 12 makes leakage canonical.

**Retraining as the reflex response to every degradation.** It was the right call for the
calibration break here and it did nothing for the objective mismatch. Diagnose the mechanism
first: label shift is fixed by fresh data, feature-pipeline skew by fixing the pipeline, and a
misspecified loss only by respecifying it.

## Interview Questions

1. **Derive this.** Show why precision@k rises when the base rate rises, holding the model's
   ranking fixed. *Answer shape:* write precision@k as the expected share of positives in the
   top-k of a fixed ranking; scaling the positive class density scales the numerator faster than
   the denominator, so the metric moves without any change in ranking quality — which is why a
   precision dashboard is not a drift detector.
2. **Design this.** PayFlow gives you a collections team that can work 300 invoices a day and
   asks for a system to fill the queue. Design it end to end. *Answer shape:* rule as floor and
   fallback; model in shadow first; selection by top-k under capacity rather than by a fixed
   threshold; objective weighted by invoice amount; calibration gate; the baseline scored forever
   alongside the model; recall tracked against delayed ground truth.
3. **Debug this.** A fraud model's precision has improved month over month for a quarter, but the
   fraud losses the team is measured on are up. Where do you look, in what order? *Answer shape:*
   the observability ladder — check whether the positive rate moved before believing the metric,
   compare predicted against observed rates for calibration, segment the label by any population
   that changed systems, then diff versions of model, features and upstream infrastructure.
4. When is a hand-written rule the *correct* production answer, rather than a placeholder?
   *Answer shape:* signal concentrated in few expert-known features; hard explainability
   constraints; no labelled history; or a measured model advantage smaller than the operating
   cost of owning the model.
5. Your model beats the incumbent by 2 points of precision. What do you need before you claim
   it is better? *Answer shape:* the same operating point, a variance estimate from repeated
   refits or folds, the business objective measured alongside the proxy, and a temporal rather
   than random split if the data has time structure.
6. Why was the feature distribution unchanged during this incident even though the model broke?
   *Answer shape:* label shift — P(y|x) changed while P(x) did not; input-validation and
   data-drift monitors that watch only features are structurally blind to it.
7. You have three months of post-migration data and three years from before it. How do you
   retrain? *Answer shape:* discuss recency weighting, a regime indicator feature, or training on
   the new regime alone; decide with a backtest on held-out post-migration rows rather than by
   preference, and note the variance cost of a smaller training set.

## Key Takeaways

- Build the dumb baseline and the incumbent rule *first*, and report every model number beside
  them; a score without a floor is not evidence.
- Compare policies at a matched operating point — same k or same threshold — or you are
  measuring appetite for queue size rather than model quality.
- Measure the business objective next to the metric you train on; here precision rose 0.425 to
  0.446 while value captured fell $51,362 to $49,665, and both were real.
- Establish the noise band before claiming a win: bootstrap refits gave ±0.0015 on precision and
  ±$437 on value, which is what makes the deltas interpretable at all.
- Statistical significance is not practical significance — a −$228 delta against a $63 band is
  detectable and irrelevant.
- Monitor calibration and recall, not only self-computable metrics like precision, which drift
  upward under label shift and hide the failure.
- Retraining repairs drift and cannot repair a misspecified objective; diagnose which one you
  have before reaching for fresh data.
- Choosing not to ship an ML system is a legitimate, defensible engineering outcome when the
  measured advantage is smaller than the cost of owning it.

## Related

**Backward** — none; 01.1 is the track's first notebook. The environment and dataset it assumes
are specified in `requirements.txt` and `_data/SPEC.md`.

**Forward**

- **01.2 First Contact with the PayFlow Data Universe** — the six tables, their grain, and the
  mess this notebook delegated to the lab module (duplicate rows, mixed currencies, two
  timestamp formats).
- **01.3 Reproducibility as an Engineering Contract** — formalizes the bootstrap-variance habit
  used in Evaluation into seeds, pins and run manifests.
- **01.5 Problem Framing** — generalizes the label choice made here (late = more than seven days
  past due) into horizon, population and cutoff as explicit design decisions.
- **09.1 Data Cleaning & Pre-processing** — owns the cleaning steps `build_dataset` performs,
  including the imputation and encoding used inside the Stage B pipeline.
- **12.1 Model Evaluation & Validation** — canonical home for baselines, cross-validation,
  leakage (the `reminder_count` trap deferred here) and meaningful-delta reasoning.
- **15.1 Logistic Regression & Classifier Practice** — derives the logistic loss used as a black
  box in Stage B, and makes thresholds, imbalance and calibration canonical.
- **20.1 Ensembles & Gradient Boosting** — the canonical home of the `HistGradientBoosting`
  strength check used here without explanation.
- **34.1 ML in Production** — turns the Stage C fallback and the prevention list into a monitored
  service with drift detection and a retraining loop.